In [1]:
import pandas as pd
import numpy as np
from scipy.optimize import curve_fit
from scipy.stats import t
import os

In [2]:
# Define the model functions
def mole_fraction_co2_model(res_time, mole_fraction_co2_eq, k):
    return mole_fraction_co2_eq - (mole_fraction_co2_eq - 1) * np.exp(-k * res_time)

In [3]:
def conv_model(res_time, conv_eq, k):
    return conv_eq - conv_eq * np.exp(-k * res_time)

In [4]:
# Constants and helper functions for energy efficiency calculation
def gibbs_free_enthalpy_CO(temp):
    return -1.48747E-20 * temp**6 + 2.90543E-16 * temp**5 - 2.18412E-12 * temp**4 + \
           7.80039E-09 * temp**3 - 1.14364E-05 * temp**2 - 8.19286E-02 * temp - 1.12455E+02

In [5]:
def gibbs_free_enthalpy_CO2(temp):
    return 1.21850E-21 * temp**6 - 2.63973E-17 * temp**5 + 2.25893E-13 * temp**4 - \
           9.56091E-10 * temp**3 + 2.75906E-06 * temp**2 - 4.68102E-03 * temp - 3.93205E+02

In [6]:
def fit_models(group):
    # Weight each point by its GC standard deviation (conv_sd). absolute_sigma=True
    # keeps the parameter covariance in real (absolute) units instead of rescaling it
    # by the reduced chi-square. This mirrors drm_conv_fit.ipynb.
    params_mole_fraction_co2, cov_mole_fraction_co2 = curve_fit(
        mole_fraction_co2_model,
        group['res_time_sec'],
        group['mole_fraction_co2'],
        p0=[0.5, 0.05],
        sigma=group['mole_fraction_co2_sd'],
        absolute_sigma=True
    )
    params_conv, cov_conv = curve_fit(
        conv_model,
        group['res_time_sec'],
        group['conv'],
        p0=[0.5, 0.05],
        sigma=group['conv_sd'],
        absolute_sigma=True
    )
    return params_mole_fraction_co2, cov_mole_fraction_co2, params_conv, cov_conv

In [7]:
def predict_sd(jac_cols, cov):
    """Standard deviation of a model prediction from the parameter covariance.

    jac_cols : list of arrays, each the partial derivative d f / d theta_i evaluated
               over the prediction grid (one entry per fitted parameter).
    cov      : parameter covariance matrix returned by curve_fit.

    Returns sqrt(J . cov . J^T) per grid point, so the full covariance (including the
    off-diagonal parameter correlation) is taken into account.
    """
    J = np.vstack(jac_cols).T              # shape (n_pred, n_params)
    return np.sqrt(np.einsum('ij,jk,ik->i', J, cov, J))


def calculate_confidence_intervals(
        res_time_range,
        params_mole_fraction_co2,
        cov_mole_fraction_co2,
        params_conv,
        cov_conv,
        n
):
    alpha = 0.05
    dof = max(0, n - len(params_mole_fraction_co2))
    tval = t.ppf(1.0 - alpha / 2., dof)

    mole_fraction_co2_eq, k = params_mole_fraction_co2
    mole_fraction_co2_eq_sd, k_sd = np.sqrt(np.diag(cov_mole_fraction_co2))

    conf_values = pd.DataFrame({'res_time_sec': res_time_range})
    conf_values['mole_fraction_co2_fit'] = mole_fraction_co2_model(res_time_range, mole_fraction_co2_eq, k)
    conf_values['conv_fit'] = conv_model(res_time_range, params_conv[0], params_conv[1])

    # Prediction band for the mole-fraction model x = x_eq - (x_eq - 1) * exp(-k t):
    #   d x / d x_eq = 1 - exp(-k t)
    #   d x / d k    = (1 - x_eq) * t * exp(-k t)
    exp_mf = np.exp(-k * res_time_range)
    conf_values['mole_fraction_co2_fit_sd'] = predict_sd(
        [1 - exp_mf, (1 - mole_fraction_co2_eq) * res_time_range * exp_mf],
        cov_mole_fraction_co2
    )
    conf_values['mole_fraction_co2_fit_lcl'] = conf_values['mole_fraction_co2_fit'] - tval * conf_values['mole_fraction_co2_fit_sd']
    conf_values['mole_fraction_co2_fit_ucl'] = conf_values['mole_fraction_co2_fit'] + tval * conf_values['mole_fraction_co2_fit_sd']

    # Equilibrium conversion derived from the equilibrium mole fraction.
    # conv_eq = (1 - x_eq) / (1 + 0.5 x_eq); both numerator and denominator depend on
    # x_eq, so use the exact single-variable derivative (the quotient cannot be treated
    # as two independent terms): d conv_eq / d x_eq = -1.5 / (1 + 0.5 x_eq)**2.
    conv_eq = (1 - mole_fraction_co2_eq) / (1 + 0.5 * mole_fraction_co2_eq)
    conv_eq_sd = 1.5 / (1 + 0.5 * mole_fraction_co2_eq)**2 * mole_fraction_co2_eq_sd

    # Prediction band for the conversion model conv = conv_eq_fit * (1 - exp(-k_conv t)),
    # using the dedicated weighted conv fit (params_conv / cov_conv):
    #   d conv / d conv_eq_fit = 1 - exp(-k_conv t)
    #   d conv / d k_conv      = conv_eq_fit * t * exp(-k_conv t)
    conv_eq_fit, k_conv = params_conv
    exp_conv = np.exp(-k_conv * res_time_range)
    conf_values['conv_fit_sd'] = predict_sd(
        [1 - exp_conv, conv_eq_fit * res_time_range * exp_conv],
        cov_conv
    )
    conf_values['conv_fit_lcl'] = conf_values['conv_fit'] - tval * conf_values['conv_fit_sd']
    conf_values['conv_fit_ucl'] = conf_values['conv_fit'] + tval * conf_values['conv_fit_sd']

    return conf_values, mole_fraction_co2_eq, mole_fraction_co2_eq_sd, conv_eq, conv_eq_sd, k, k_sd

In [8]:
def energy_efficiency_calculations(conf_values, group):
    power_avg = group['plasma_power_watt_avg'].mean()
    reactor_vol_ml = 17.31
    packing_factor = 0.4774
    conc_co2_blank = 1
    gas_temp_avg = group['gas_temp_avg_dgc'].mean()
    gas_temp = 273.15 + gas_temp_avg
    
    gf_co = gibbs_free_enthalpy_CO(gas_temp)
    gf_co2 = gibbs_free_enthalpy_CO2(gas_temp)
    
    conf_values['co2_flux_fit'] = (reactor_vol_ml * (1 - packing_factor)) / (conf_values['res_time_sec'] / 60)
    conf_values['alpha_fit'] = 1 + (1 / 2) * conf_values['conv_fit']
    conf_values['alpha_fit_sd'] = (1 / 2) * conf_values['conv_fit'] * conf_values['conv_fit_sd'] / conf_values['conv_fit']
    
    conf_values['conc_co_fit'] = conf_values['conv_fit'] / (1 + (1 / 2) * conf_values['conv_fit'])
    conf_values['conc_co_fit_sd'] = conf_values['conc_co_fit'] * np.sqrt(
        conf_values['conv_fit_sd']**2 / conf_values['conv_fit']**2 +
        (0.5 * conf_values['conv_fit_sd'] / (1 + 0.5 * conf_values['conv_fit']))**2
    )
    
    conf_values['conc_o2_fit'] = conf_values['conv_fit'] / (2 + conf_values['conv_fit'])
    conf_values['conc_o2_fit_sd'] = conf_values['conc_o2_fit'] * np.sqrt(
        conf_values['conv_fit_sd']**2 / conf_values['conv_fit']**2 +
        (conf_values['conv_fit_sd'] / (2 + conf_values['conv_fit']))**2
    )
    
    conf_values['sei_fit'] = (power_avg / conf_values['co2_flux_fit']) * 60 * 24.055
    
    conf_values['ee_gf_fit'] = (conf_values['alpha_fit'] * conf_values['conc_co_fit'] * gf_co -
                                conf_values['conv_fit'] * conc_co2_blank * gf_co2) / conf_values['sei_fit'] * 100
    conf_values['ee_gf_fit_sd'] = 100 * np.sqrt(
        (conf_values['alpha_fit_sd'] / conf_values['alpha_fit'] * conf_values['alpha_fit'] * conf_values['conc_co_fit'] * gf_co)**2 +
        (conf_values['conv_fit_sd'] * conc_co2_blank * gf_co2)**2
    ) / conf_values['sei_fit']
    
    conf_values['ee_gf_fit_lcl'] = conf_values['ee_gf_fit'] - t.ppf(1.0 - 0.05 / 2., len(group) - 1) * conf_values['ee_gf_fit_sd']
    conf_values['ee_gf_fit_ucl'] = conf_values['ee_gf_fit'] + t.ppf(1.0 - 0.05 / 2., len(group) - 1) * conf_values['ee_gf_fit_sd']
    
    return conf_values

In [9]:
def flag_outliers(group, model, p0, z_thresh=3.5, scale_floor=0.01, max_iter=5):
    """Iterative studentized-residual outlier detection for one packing group.

    Strategy (kept deliberately robust against the data-quality issues in this dataset):
      * Detection uses an **unweighted** (ordinary) fit. A genuinely bad point sometimes
        carries an artificially tiny conv_sd; a weighted fit would then be captured by it
        and hide its own residual, so the GC weights are used only for the *final* fit
        (fit_models), not here.
      * Exact duplicate measurement rows (the same GC result entered several times) are
        collapsed to unique (res_time_sec, conv) points so they neither dominate the fit
        nor collapse the residual scale to zero.
      * Residuals are standardised by a robust MAD scale (1.4826 * MAD) floored at
        ``scale_floor`` conversion units. The floor stops a near-perfect first-order fit
        from flagging harmless model curvature (a few-thousandths deviation) as an outlier.
      * Points with |studentized residual| > ``z_thresh`` are flagged, removed, and the fit
        is repeated until nothing new is flagged (or ``max_iter`` is reached).

    ``z_thresh`` and ``scale_floor`` are the two knobs; defaults flag only gross outliers.
    Returns the group with an added boolean ``is_outlier`` column (set on every duplicate
    row of a flagged measurement).
    """
    grp = group.copy().reset_index(drop=True)
    uniq = grp.drop_duplicates(subset=['res_time_sec', 'conv']).reset_index(drop=True)
    out = np.zeros(len(uniq), dtype=bool)
    for _ in range(max_iter):
        keep = ~out
        if keep.sum() <= len(p0) + 1:           # too few points left to fit meaningfully
            break
        try:
            popt, _ = curve_fit(model, uniq.loc[keep, 'res_time_sec'], uniq.loc[keep, 'conv'], p0=p0)
        except RuntimeError:
            break
        resid = uniq['conv'].values - model(uniq['res_time_sec'].values, *popt)
        med = np.median(resid[keep])
        scale = max(1.4826 * np.median(np.abs(resid[keep] - med)), scale_floor)
        z = (resid - med) / scale
        new = (np.abs(z) > z_thresh) & keep
        if not new.any():
            break
        out[new] = True
    # Map the per-measurement flags back onto every (possibly duplicated) row.
    bad = set(map(tuple, np.round(uniq.loc[out, ['res_time_sec', 'conv']].values, 10)))
    keys = map(tuple, np.round(grp[['res_time_sec', 'conv']].values, 10))
    grp['is_outlier'] = [k in bad for k in keys]
    return grp

In [ ]:
# Load the full reaction data (all compounds; only the plasma state is present).
parent_folder = r'C:\Users\Sande\OneDrive - Universiteit Antwerpen\plasmacatdesign\co2-splitting\uhasselt'
file_path = os.path.join(parent_folder, 'uhasselt_co2_spl_reaction_data_combined.csv')
raw = pd.read_csv(file_path)

# --- CO2 conversion working frame (the fitted quantity) -------------------------
data = raw[raw['compound'] == 'CO2'].copy()
data['mole_fraction_co2'] = (1 - data['conv']) / (1 + 1 / 2 * data['conv'])

# conv_sd is the GC-run standard deviation (the fit weight). Floor missing /
# non-positive values with the group median, then the global median.
data['conv_sd'] = data['conv_sd'].where(data['conv_sd'] > 0)
data['conv_sd'] = data['conv_sd'].fillna(data.groupby('packing')['conv_sd'].transform('median'))
data['conv_sd'] = data['conv_sd'].fillna(data['conv_sd'].median())
data['mole_fraction_co2_sd'] = 1.5 / (1 + 0.5 * data['conv']) ** 2 * data['conv_sd']

# --- Measured production points per compound (overlaid on the fits) -------------
# CO2 conversion = `conv` (CO2 rows); CO/O2 production = carbon/oxygen yield on the
# CO/O2 rows. These are independent GC measurements, plotted *without* error bars
# (one GC run per reactor run -> pseudo-replication); their SDs are kept only as a
# weight. `measurement_number` identifies the reactor run.
measured = {
    'CO2': raw.loc[raw['compound'] == 'CO2',
                   ['packing', 'res_time_sec', 'measurement_number', 'conv', 'conv_sd']]
              .rename(columns={'conv': 'value_measured', 'conv_sd': 'value_measured_sd'}),
    'CO':  raw.loc[raw['compound'] == 'CO',
                   ['packing', 'res_time_sec', 'measurement_number', 'yield_c', 'yield_c_sd']]
              .rename(columns={'yield_c': 'value_measured', 'yield_c_sd': 'value_measured_sd'}),
    'O2':  raw.loc[raw['compound'] == 'O2',
                   ['packing', 'res_time_sec', 'measurement_number', 'yield_o', 'yield_o_sd']]
              .rename(columns={'yield_o': 'value_measured', 'yield_o_sd': 'value_measured_sd'}),
}


def weighted_r2(y, pred, w=None):
    # Coefficient of determination (optionally weighted) of measured `y` vs the
    # model prediction `pred` evaluated at the same residence times.
    y = np.asarray(y, float)
    pred = np.asarray(pred, float)
    m = np.isfinite(y) & np.isfinite(pred)
    y, pred = y[m], pred[m]
    w = np.ones_like(y) if w is None else np.asarray(w, float)[m]
    if len(y) < 2:
        return np.nan
    wmean = np.sum(w * y) / np.sum(w)
    ssr = np.sum(w * (y - pred) ** 2)
    tss = np.sum(w * (y - wmean) ** 2)
    return 1 - ssr / tss if tss > 0 else np.nan


# CO2-fit-derived energy-efficiency / mole-fraction columns kept on the CO2 rows.
co2_only_cols = [
    'mole_fraction_co2_fit', 'mole_fraction_co2_fit_sd',
    'mole_fraction_co2_fit_lcl', 'mole_fraction_co2_fit_ucl',
    'mole_fraction_co2_eq', 'mole_fraction_co2_eq_sd',
    'co2_flux_fit', 'alpha_fit', 'alpha_fit_sd',
    'conc_co_fit', 'conc_co_fit_sd', 'conc_o2_fit', 'conc_o2_fit_sd',
    'ee_gf_fit', 'ee_gf_fit_sd', 'ee_gf_fit_lcl', 'ee_gf_fit_ucl',
]

res_time_range = np.linspace(1, 100, 199)
all_results = []
n_outliers = 0

for packing, group in data.groupby('packing'):
    # Flag outlier reactor runs on the CO2 conversion, then fit on retained points.
    group = flag_outliers(group, conv_model, p0=[0.5, 0.05])
    n_outliers += int(group['is_outlier'].sum())
    fit_group = group[~group['is_outlier']]

    params_mf, cov_mf, params_conv, cov_conv = fit_models(fit_group)
    conf_values, mf_eq, mf_eq_sd, conv_eq, conv_eq_sd, k, k_sd = calculate_confidence_intervals(
        res_time_range, params_mf, cov_mf, params_conv, cov_conv, len(fit_group))
    conf_values = energy_efficiency_calculations(conf_values, fit_group)
    conf_values['packing'] = packing
    conf_values['mole_fraction_co2_eq'] = mf_eq
    conf_values['mole_fraction_co2_eq_sd'] = mf_eq_sd
    conf_values['conv_eq'] = conv_eq
    conf_values['conv_eq_sd'] = conv_eq_sd
    conf_values['k'] = k
    conf_values['k_sd'] = k_sd

    sei_lookup = conf_values.set_index('res_time_sec')['sei_fit']
    bad_runs = set(map(tuple, group.loc[group['is_outlier'],
                                        ['res_time_sec', 'measurement_number']].values))

    # CO2 weighted R^2 (fit vs measured, non-outlier, weighted by 1/conv_sd^2).
    pred_co2 = conv_model(fit_group['res_time_sec'].values, *params_conv)
    r2_co2 = weighted_r2(fit_group['conv'].values, pred_co2, 1 / fit_group['conv_sd'].values ** 2)
    n_co2 = len(fit_group)

    # Stoichiometric production fits (CO2 -> CO + 1/2 O2): CO yield = conversion,
    # O2 yield = 1/2 conversion. Curves and bands are transforms of the CO2 conv fit;
    # measured CO/O2 yields are overlaid to show how well stoichiometry closes.
    compound_specs = {
        'CO2': dict(scale=1.0, eq=conv_eq, eq_sd=conv_eq_sd),
        'CO':  dict(scale=1.0, eq=conv_eq, eq_sd=conv_eq_sd),
        'O2':  dict(scale=0.5, eq=0.5 * conv_eq, eq_sd=0.5 * conv_eq_sd),
    }

    for comp, spec in compound_specs.items():
        s = spec['scale']
        m = measured[comp]
        m = m[m['packing'] == packing].dropna(subset=['value_measured']).copy()
        # Explicit boolean dtype so the empty-group case keeps a valid boolean mask.
        m['is_outlier'] = np.array(
            [tuple(r) in bad_runs
             for r in m[['res_time_sec', 'measurement_number']].values],
            dtype=bool,
        )

        if comp == 'CO2':
            r2, n_pts = r2_co2, n_co2
        else:
            mk = m[~m['is_outlier']]
            pred_m = s * conv_model(mk['res_time_sec'].values, *params_conv)
            r2 = weighted_r2(mk['value_measured'].values, pred_m)
            n_pts = int(np.isfinite(mk['value_measured'].values).sum())

        fit_rows = pd.DataFrame({
            'packing': packing,
            'compound': comp,
            'res_time_sec': conf_values['res_time_sec'].values,
            'sei_fit': conf_values['sei_fit'].values,
            'row_type': 'fit',
            'value_fit': s * conf_values['conv_fit'].values,
            'value_fit_sd': s * conf_values['conv_fit_sd'].values,
            'value_fit_lcl': s * conf_values['conv_fit_lcl'].values,
            'value_fit_ucl': s * conf_values['conv_fit_ucl'].values,
            'value_eq': spec['eq'],
            'value_eq_sd': spec['eq_sd'],
            'k': k, 'k_sd': k_sd,
            'r_squared': r2, 'n_points': n_pts,
            'value_measured': np.nan, 'value_measured_sd': np.nan,
            'is_outlier': False, 'measurement_number': np.nan,
        })
        if comp == 'CO2':
            for col in co2_only_cols:
                fit_rows[col] = conf_values[col].values

        meas_rows = pd.DataFrame({
            'packing': packing,
            'compound': comp,
            'res_time_sec': m['res_time_sec'].values,
            'sei_fit': m['res_time_sec'].map(sei_lookup).values,
            'row_type': 'measured',
            'value_fit': np.nan, 'value_fit_sd': np.nan,
            'value_fit_lcl': np.nan, 'value_fit_ucl': np.nan,
            'value_eq': spec['eq'], 'value_eq_sd': spec['eq_sd'],
            'k': k, 'k_sd': k_sd,
            'r_squared': r2, 'n_points': n_pts,
            'value_measured': m['value_measured'].values,
            'value_measured_sd': m['value_measured_sd'].values,
            'is_outlier': m['is_outlier'].values,
            'measurement_number': m['measurement_number'].values,
        })

        all_results.append(pd.concat([fit_rows, meas_rows], ignore_index=True))

# Combine all results into a single long-over-compound DataFrame.
final_results = pd.concat(all_results, ignore_index=True)

output_path = os.path.join(parent_folder, 'uhasselt_co2_spl_reaction_fitting_results.csv')
final_results.to_csv(output_path, index=False)

print("Fitting results saved to:", output_path)
print("Rows:", len(final_results),
      "| compounds:", sorted(final_results['compound'].unique().tolist()))
print("Total CO2 reactor-run points flagged as outliers:", n_outliers)